# SGB: Sistema de Gestión de Biblioteca (versión final, interfaz en Gradio)
Programación de Computadores 2, Ingeniería de Software.

Versión final con los prototipos P0 a P5: base Colab, usuarios y libros, préstamos, reservas y multas, análisis de datos e integración.
La lógica (modelo, repositorios y servicios) usa solo la librería estándar de Python. La sección 4 es la capa de presentación con Gradio.

**Cómo usarlo:** ejecutar todas las celdas en orden (Entorno de ejecución > Ejecutar todo). La interfaz aparece debajo de la última celda.
Los datos de prueba se cargan solos al ejecutar la celda 5.3 (variable `CARGAR_DATOS_DE_PRUEBA`).

## Celda 0: configuración (Drive y rutas)

In [35]:
import os
from datetime import datetime, date, timedelta

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_BASE = '/content/drive/MyDrive/SGB/datos/'
except ImportError:
    # Ejecución fuera de Colab (pruebas locales)
    RUTA_BASE = './SGB/datos/'

os.makedirs(RUTA_BASE, exist_ok=True)

RUTA_USUARIOS   = RUTA_BASE + 'usuarios.txt'
RUTA_LIBROS     = RUTA_BASE + 'libros.txt'
RUTA_CATEGORIAS = RUTA_BASE + 'categorias.txt'
RUTA_PRESTAMOS  = RUTA_BASE + 'prestamos.txt'
RUTA_MULTAS     = RUTA_BASE + 'multas.txt'
RUTA_RESERVAS   = RUTA_BASE + 'reservas.txt'

NINGUNO = 'NINGUNO'
FORMATO_FECHA = '%Y-%m-%d'
print('Rutas listas en:', RUTA_BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rutas listas en: /content/drive/MyDrive/SGB/datos/


## Celda 0.1: biblioteca Gradio

In [36]:
try:
    import gradio as gr
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'])
    import gradio as gr
print('Gradio', gr.__version__)

Gradio 6.31.0


## Sección 1: modelo (dominio)

### Celda 1.1: enums y reglas

In [37]:
from enum import Enum


class TipoUsuarioEnum(Enum):
    """Tipos de usuario de la biblioteca."""
    ESTUDIANTE = 'ESTUDIANTE'
    DOCENTE = 'DOCENTE'
    EXTERNO = 'EXTERNO'


class EstadoPrestamoEnum(Enum):
    """Estados posibles de un préstamo."""
    ACTIVO = 'ACTIVO'
    DEVUELTO = 'DEVUELTO'
    VENCIDO = 'VENCIDO'


class EstadoMultaEnum(Enum):
    """Estados posibles de una multa."""
    PENDIENTE = 'PENDIENTE'
    PAGADA = 'PAGADA'


class EstadoReservaEnum(Enum):
    """Estados posibles de una reserva."""
    ACTIVA = 'ACTIVA'
    CUMPLIDA = 'CUMPLIDA'
    CANCELADA = 'CANCELADA'


# Reglas por tipo de usuario (RF-03): días de préstamo y cupo de libros
DIAS_PRESTAMO = {
    TipoUsuarioEnum.ESTUDIANTE: 7,
    TipoUsuarioEnum.DOCENTE: 15,
    TipoUsuarioEnum.EXTERNO: 5,
}
CUPO_LIBROS = {
    TipoUsuarioEnum.ESTUDIANTE: 3,
    TipoUsuarioEnum.DOCENTE: 5,
    TipoUsuarioEnum.EXTERNO: 2,
}
MULTA_POR_DIA = 1000          # COP por cada día de retraso (RF-04)
ESTADO_USUARIO_ACTIVO = 'ACTIVO'
ESTADO_USUARIO_SUSPENDIDO = 'SUSPENDIDO'

### Celda 1.2: clase Usuario

In [38]:
class Usuario:
    """Persona registrada en la biblioteca."""

    def __init__(self, num_documento, nombre, tipo, correo, telefono,
                 estado=ESTADO_USUARIO_ACTIVO):
        self.num_documento = num_documento
        self.nombre = nombre
        self.tipo = tipo            # TipoUsuarioEnum
        self.correo = correo
        self.telefono = telefono
        self.estado = estado        # 'ACTIVO' o 'SUSPENDIDO'

    def to_linea(self):
        """Convierte el usuario a una línea de texto delimitada por |."""
        return '|'.join([self.num_documento, self.nombre, self.tipo.value,
                         self.correo, self.telefono, self.estado])

    @staticmethod
    def from_linea(linea):
        """Construye un Usuario a partir de una línea de usuarios.txt."""
        c = linea.split('|')
        return Usuario(c[0], c[1], TipoUsuarioEnum(c[2]), c[3], c[4], c[5])

    def __str__(self):
        return (f'{self.num_documento} | {self.nombre} | {self.tipo.value} | '
                f'{self.correo} | {self.telefono} | {self.estado}')

### Celda 1.3: clase Categoria

In [39]:
class Categoria:
    """Categoría temática que agrupa libros."""

    def __init__(self, codigo, nombre, descripcion):
        self.codigo = codigo
        self.nombre = nombre
        self.descripcion = descripcion

    def to_linea(self):
        """Convierte la categoría a una línea delimitada por |."""
        return '|'.join([self.codigo, self.nombre, self.descripcion])

    @staticmethod
    def from_linea(linea):
        """Construye una Categoria a partir de una línea de categorias.txt."""
        c = linea.split('|')
        return Categoria(c[0], c[1], c[2])

    def __str__(self):
        return f'{self.codigo} | {self.nombre} | {self.descripcion}'

### Celda 1.4: clase Libro

In [40]:
class Libro:
    """Libro del catálogo con control de ejemplares."""

    def __init__(self, isbn, titulo, autor, cod_categoria, anio,
                 total_ejemplares, disponibles):
        self.isbn = isbn
        self.titulo = titulo
        self.autor = autor
        self.cod_categoria = cod_categoria
        self.anio = anio                          # int
        self.total_ejemplares = total_ejemplares  # int
        self.disponibles = disponibles            # int

    def to_linea(self):
        """Convierte el libro a una línea delimitada por |."""
        return '|'.join([self.isbn, self.titulo, self.autor, self.cod_categoria,
                         str(self.anio), str(self.total_ejemplares),
                         str(self.disponibles)])

    @staticmethod
    def from_linea(linea):
        """Construye un Libro a partir de una línea de libros.txt."""
        c = linea.split('|')
        return Libro(c[0], c[1], c[2], c[3], int(c[4]), int(c[5]), int(c[6]))

    def __str__(self):
        return (f'{self.isbn} | {self.titulo} | {self.autor} | cat: {self.cod_categoria} | '
                f'{self.anio} | disponibles: {self.disponibles}/{self.total_ejemplares}')

### Celda 1.5: clase Prestamo

In [41]:
class Prestamo:
    """Préstamo de un libro a un usuario."""

    def __init__(self, codigo, isbn, num_documento, fecha_prestamo, fecha_limite,
                 fecha_devolucion=NINGUNO, estado=EstadoPrestamoEnum.ACTIVO):
        self.codigo = codigo
        self.isbn = isbn
        self.num_documento = num_documento
        self.fecha_prestamo = fecha_prestamo      # texto AAAA-MM-DD
        self.fecha_limite = fecha_limite          # texto AAAA-MM-DD
        self.fecha_devolucion = fecha_devolucion  # texto AAAA-MM-DD o NINGUNO
        self.estado = estado                      # EstadoPrestamoEnum

    def to_linea(self):
        """Convierte el préstamo a una línea delimitada por |."""
        return '|'.join([self.codigo, self.isbn, self.num_documento,
                         self.fecha_prestamo, self.fecha_limite,
                         self.fecha_devolucion, self.estado.value])

    @staticmethod
    def from_linea(linea):
        """Construye un Prestamo a partir de una línea de prestamos.txt."""
        c = linea.split('|')
        return Prestamo(c[0], c[1], c[2], c[3], c[4], c[5], EstadoPrestamoEnum(c[6]))

    def __str__(self):
        return (f'{self.codigo} | libro {self.isbn} | usuario {self.num_documento} | '
                f'prestado {self.fecha_prestamo} | límite {self.fecha_limite} | '
                f'devuelto {self.fecha_devolucion} | {self.estado.value}')

### Celda 1.6: clase Multa

In [42]:
class Multa:
    """Multa generada por la devolución tardía de un préstamo."""

    def __init__(self, codigo, cod_prestamo, dias_retraso, valor,
                 estado=EstadoMultaEnum.PENDIENTE):
        self.codigo = codigo
        self.cod_prestamo = cod_prestamo
        self.dias_retraso = dias_retraso   # int
        self.valor = valor                 # int, en COP
        self.estado = estado               # EstadoMultaEnum

    def to_linea(self):
        """Convierte la multa a una línea delimitada por |."""
        return '|'.join([self.codigo, self.cod_prestamo, str(self.dias_retraso),
                         str(self.valor), self.estado.value])

    @staticmethod
    def from_linea(linea):
        """Construye una Multa a partir de una línea de multas.txt."""
        c = linea.split('|')
        return Multa(c[0], c[1], int(c[2]), int(c[3]), EstadoMultaEnum(c[4]))

    def __str__(self):
        return (f'{self.codigo} | préstamo {self.cod_prestamo} | {self.dias_retraso} día(s) | '
                f'COP {self.valor} | {self.estado.value}')

### Celda 1.7: clase Reserva

In [43]:
class Reserva:
    """Reserva de un libro sin ejemplares disponibles."""

    def __init__(self, codigo, isbn, num_documento, fecha_reserva,
                 estado=EstadoReservaEnum.ACTIVA):
        self.codigo = codigo
        self.isbn = isbn
        self.num_documento = num_documento
        self.fecha_reserva = fecha_reserva   # texto AAAA-MM-DD
        self.estado = estado                 # EstadoReservaEnum

    def to_linea(self):
        """Convierte la reserva a una línea delimitada por |."""
        return '|'.join([self.codigo, self.isbn, self.num_documento,
                         self.fecha_reserva, self.estado.value])

    @staticmethod
    def from_linea(linea):
        """Construye una Reserva a partir de una línea de reservas.txt."""
        c = linea.split('|')
        return Reserva(c[0], c[1], c[2], c[3], EstadoReservaEnum(c[4]))

    def __str__(self):
        return (f'{self.codigo} | libro {self.isbn} | usuario {self.num_documento} | '
                f'{self.fecha_reserva} | {self.estado.value}')

## Sección 2: repositorios (persistencia)

### Celda 2.1: ArchivoUtil

In [44]:
class ArchivoUtil:
    """Lectura y escritura de archivos de texto (un registro por línea)."""

    @staticmethod
    def leer_lineas(ruta):
        """Lee todas las líneas de un archivo. Retorna lista vacía si no existe."""
        try:
            with open(ruta, 'r', encoding='utf-8') as f:
                return [l.strip() for l in f if l.strip()]
        except FileNotFoundError:
            return []

    @staticmethod
    def escribir_linea(ruta, linea):
        """Agrega una línea al final del archivo."""
        with open(ruta, 'a', encoding='utf-8') as f:
            f.write(linea + '\n')

    @staticmethod
    def reescribir(ruta, lineas):
        """Reemplaza el contenido completo del archivo."""
        with open(ruta, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lineas) + ('\n' if lineas else ''))

    @staticmethod
    def generar_codigo(prefijo):
        """Genera un código único basado en timestamp."""
        ts = datetime.now().strftime('%Y%m%d%H%M%S%f')[:20]
        return f'{prefijo}-{ts}'

    @staticmethod
    def reemplazar_linea(ruta, clave, nueva_linea):
        """Reemplaza la línea cuyo primer campo es clave. Retorna True si la encontró."""
        lineas = ArchivoUtil.leer_lineas(ruta)
        encontrada = False
        for i, l in enumerate(lineas):
            if l.split('|')[0] == clave:
                lineas[i] = nueva_linea
                encontrada = True
                break
        if encontrada:
            ArchivoUtil.reescribir(ruta, lineas)
        return encontrada

### Celda 2.2: UsuarioRepository

In [45]:
class UsuarioRepository:
    """Persistencia de usuarios en usuarios.txt."""

    def __init__(self, ruta=RUTA_USUARIOS):
        self.ruta = ruta

    def guardar(self, u):
        """Agrega un usuario al archivo."""
        ArchivoUtil.escribir_linea(self.ruta, u.to_linea())

    def find_by_documento(self, doc):
        """Retorna el Usuario con ese documento, o None."""
        for l in ArchivoUtil.leer_lineas(self.ruta):
            if l.split('|')[0] == doc:
                return Usuario.from_linea(l)
        return None

    def find_all(self):
        """Retorna la lista de todos los usuarios."""
        return [Usuario.from_linea(l) for l in ArchivoUtil.leer_lineas(self.ruta)]

    def actualizar(self, u):
        """Reemplaza en el archivo el registro del usuario."""
        return ArchivoUtil.reemplazar_linea(self.ruta, u.num_documento, u.to_linea())

### Celda 2.3: CategoriaRepository

In [46]:
class CategoriaRepository:
    """Persistencia de categorías en categorias.txt."""

    def __init__(self, ruta=RUTA_CATEGORIAS):
        self.ruta = ruta

    def guardar(self, c):
        """Agrega una categoría al archivo."""
        ArchivoUtil.escribir_linea(self.ruta, c.to_linea())

    def find_by_codigo(self, cod):
        """Retorna la Categoria con ese código, o None."""
        for l in ArchivoUtil.leer_lineas(self.ruta):
            if l.split('|')[0] == cod:
                return Categoria.from_linea(l)
        return None

    def find_all(self):
        """Retorna la lista de todas las categorías."""
        return [Categoria.from_linea(l) for l in ArchivoUtil.leer_lineas(self.ruta)]

### Celda 2.4: LibroRepository

In [47]:
class LibroRepository:
    """Persistencia de libros en libros.txt."""

    def __init__(self, ruta=RUTA_LIBROS):
        self.ruta = ruta

    def guardar(self, l):
        """Agrega un libro al archivo."""
        ArchivoUtil.escribir_linea(self.ruta, l.to_linea())

    def find_by_isbn(self, isbn):
        """Retorna el Libro con ese ISBN, o None."""
        for linea in ArchivoUtil.leer_lineas(self.ruta):
            if linea.split('|')[0] == isbn:
                return Libro.from_linea(linea)
        return None

    def find_by_titulo(self, txt):
        """Retorna los libros cuyo título contiene txt (sin distinguir mayúsculas)."""
        txt = txt.lower()
        return [l for l in self.find_all() if txt in l.titulo.lower()]

    def find_by_categoria(self, cod):
        """Retorna los libros de una categoría."""
        return [l for l in self.find_all() if l.cod_categoria == cod]

    def find_all(self):
        """Retorna la lista de todos los libros."""
        return [Libro.from_linea(l) for l in ArchivoUtil.leer_lineas(self.ruta)]

    def actualizar(self, l):
        """Reemplaza en el archivo el registro del libro."""
        return ArchivoUtil.reemplazar_linea(self.ruta, l.isbn, l.to_linea())

### Celda 2.5: PrestamoRepository

In [48]:
class PrestamoRepository:
    """Persistencia de préstamos en prestamos.txt."""

    def __init__(self, ruta=RUTA_PRESTAMOS):
        self.ruta = ruta

    def guardar(self, p):
        """Agrega un préstamo al archivo."""
        ArchivoUtil.escribir_linea(self.ruta, p.to_linea())

    def find_by_codigo(self, cod):
        """Retorna el Prestamo con ese código, o None."""
        for l in ArchivoUtil.leer_lineas(self.ruta):
            if l.split('|')[0] == cod:
                return Prestamo.from_linea(l)
        return None

    def find_all(self):
        """Retorna la lista de todos los préstamos."""
        return [Prestamo.from_linea(l) for l in ArchivoUtil.leer_lineas(self.ruta)]

    def find_by_usuario(self, doc):
        """Retorna los préstamos de un usuario."""
        return [p for p in self.find_all() if p.num_documento == doc]

    def find_by_libro(self, isbn):
        """Retorna los préstamos de un libro."""
        return [p for p in self.find_all() if p.isbn == isbn]

    def actualizar(self, p):
        """Reemplaza en el archivo el registro del préstamo."""
        return ArchivoUtil.reemplazar_linea(self.ruta, p.codigo, p.to_linea())

### Celda 2.6: MultaRepository

In [49]:
class MultaRepository:
    """Persistencia de multas en multas.txt.

    La multa solo guarda el código del préstamo que la originó, por eso este
    repositorio consulta PrestamoRepository para ubicar las multas de un usuario.
    """

    def __init__(self, ruta=RUTA_MULTAS, prestamo_repo=None):
        self.ruta = ruta
        self.prestamo_repo = prestamo_repo or PrestamoRepository()

    def guardar(self, m):
        """Agrega una multa al archivo."""
        ArchivoUtil.escribir_linea(self.ruta, m.to_linea())

    def find_by_codigo(self, cod):
        """Retorna la Multa con ese código, o None."""
        for l in ArchivoUtil.leer_lineas(self.ruta):
            if l.split('|')[0] == cod:
                return Multa.from_linea(l)
        return None

    def find_all(self):
        """Retorna la lista de todas las multas."""
        return [Multa.from_linea(l) for l in ArchivoUtil.leer_lineas(self.ruta)]

    def find_by_prestamo(self, cod_prestamo):
        """Retorna la multa originada por un préstamo, o None."""
        for m in self.find_all():
            if m.cod_prestamo == cod_prestamo:
                return m
        return None

    def find_by_usuario(self, doc):
        """Retorna todas las multas de un usuario."""
        codigos = {p.codigo for p in self.prestamo_repo.find_by_usuario(doc)}
        return [m for m in self.find_all() if m.cod_prestamo in codigos]

    def find_pendientes_usuario(self, doc):
        """Retorna las multas PENDIENTE de un usuario."""
        return [m for m in self.find_by_usuario(doc) if m.estado == EstadoMultaEnum.PENDIENTE]

    def actualizar(self, m):
        """Reemplaza en el archivo el registro de la multa."""
        return ArchivoUtil.reemplazar_linea(self.ruta, m.codigo, m.to_linea())

### Celda 2.7: ReservaRepository

In [50]:
class ReservaRepository:
    """Persistencia de reservas en reservas.txt (el orden del archivo es el orden de llegada)."""

    def __init__(self, ruta=RUTA_RESERVAS):
        self.ruta = ruta

    def guardar(self, r):
        """Agrega una reserva al archivo."""
        ArchivoUtil.escribir_linea(self.ruta, r.to_linea())

    def find_by_codigo(self, cod):
        """Retorna la Reserva con ese código, o None."""
        for l in ArchivoUtil.leer_lineas(self.ruta):
            if l.split('|')[0] == cod:
                return Reserva.from_linea(l)
        return None

    def find_all(self):
        """Retorna la lista de todas las reservas."""
        return [Reserva.from_linea(l) for l in ArchivoUtil.leer_lineas(self.ruta)]

    def find_by_libro(self, isbn):
        """Retorna las reservas de un libro en orden de llegada."""
        return [r for r in self.find_all() if r.isbn == isbn]

    def find_by_usuario(self, doc):
        """Retorna las reservas de un usuario."""
        return [r for r in self.find_all() if r.num_documento == doc]

    def find_activas_libro(self, isbn):
        """Retorna las reservas ACTIVA de un libro en orden de llegada."""
        return [r for r in self.find_by_libro(isbn) if r.estado == EstadoReservaEnum.ACTIVA]

    def actualizar(self, r):
        """Reemplaza en el archivo el registro de la reserva."""
        return ArchivoUtil.reemplazar_linea(self.ruta, r.codigo, r.to_linea())

## Sección 3: servicios (lógica)

### Celda 3.1: UsuarioService

In [51]:
class UsuarioService:
    """Reglas de negocio para la gestión de usuarios (RF-01)."""

    def __init__(self, repo):
        self.repo = repo

    def registrar(self, num_documento, nombre, tipo, correo, telefono):
        """Registra un usuario nuevo. Lanza ValueError si el documento ya existe."""
        if self.repo.find_by_documento(num_documento):
            raise ValueError('Ya existe un usuario con ese documento.')
        u = Usuario(num_documento, nombre, tipo, correo, telefono)
        self.repo.guardar(u)
        return u

    def buscar_por_documento(self, doc):
        """Retorna el usuario con ese documento, o None."""
        return self.repo.find_by_documento(doc)

    def actualizar(self, doc, correo=None, telefono=None, estado=None):
        """Actualiza correo, teléfono o estado de un usuario existente."""
        u = self.repo.find_by_documento(doc)
        if u is None:
            raise ValueError('No existe un usuario con ese documento.')
        if estado and estado not in (ESTADO_USUARIO_ACTIVO, ESTADO_USUARIO_SUSPENDIDO):
            raise ValueError('Estado inválido. Use ACTIVO o SUSPENDIDO.')
        if correo:
            u.correo = correo
        if telefono:
            u.telefono = telefono
        if estado:
            u.estado = estado
        self.repo.actualizar(u)
        return u

    def listar_todos(self):
        """Retorna todos los usuarios registrados."""
        return self.repo.find_all()

### Celda 3.2: LibroService

In [52]:
class LibroService:
    """Reglas de negocio para libros y categorías (RF-02)."""

    def __init__(self, libro_repo, categoria_repo):
        self.libro_repo = libro_repo
        self.categoria_repo = categoria_repo

    def registrar(self, isbn, titulo, autor, cod_categoria, anio, total_ejemplares):
        """Registra un libro nuevo con todos sus ejemplares disponibles."""
        if self.libro_repo.find_by_isbn(isbn):
            raise ValueError('Ya existe un libro con ese ISBN.')
        if self.categoria_repo.find_by_codigo(cod_categoria) is None:
            raise ValueError('La categoría indicada no existe.')
        if total_ejemplares < 1:
            raise ValueError('El número de ejemplares debe ser al menos 1.')
        libro = Libro(isbn, titulo, autor, cod_categoria, anio,
                      total_ejemplares, total_ejemplares)
        self.libro_repo.guardar(libro)
        return libro

    def registrar_categoria(self, codigo, nombre, descripcion):
        """Registra una categoría nueva."""
        if self.categoria_repo.find_by_codigo(codigo):
            raise ValueError('Ya existe una categoría con ese código.')
        c = Categoria(codigo, nombre, descripcion)
        self.categoria_repo.guardar(c)
        return c

    def buscar_por_isbn(self, isbn):
        """Retorna el libro con ese ISBN, o None."""
        return self.libro_repo.find_by_isbn(isbn)

    def buscar_por_titulo(self, txt):
        """Retorna los libros cuyo título contiene txt."""
        return self.libro_repo.find_by_titulo(txt)

    def listar_por_categoria(self, cod):
        """Retorna los libros de una categoría."""
        return self.libro_repo.find_by_categoria(cod)

    def listar_categorias(self):
        """Retorna todas las categorías."""
        return self.categoria_repo.find_all()

    def listar_todos(self):
        """Retorna todos los libros del catálogo."""
        return self.libro_repo.find_all()

    def listar_disponibles(self):
        """Retorna los libros con al menos un ejemplar disponible."""
        return [l for l in self.libro_repo.find_all() if l.disponibles > 0]

### Celda 3.3: MultaService

In [53]:
class MultaService:
    """Reglas de negocio para multas por devolución tardía (RF-04)."""

    def __init__(self, multa_repo, prestamo_repo):
        self.multa_repo = multa_repo
        self.prestamo_repo = prestamo_repo

    def generar(self, cod_prestamo, dias_retraso):
        """Crea la multa de un préstamo (COP 1.000 por día). Máximo una por préstamo."""
        if dias_retraso <= 0:
            raise ValueError('No hay días de retraso para multar.')
        if self.prestamo_repo.find_by_codigo(cod_prestamo) is None:
            raise ValueError('El préstamo no existe.')
        if self.multa_repo.find_by_prestamo(cod_prestamo) is not None:
            raise ValueError('El préstamo ya tiene una multa.')
        multa = Multa(ArchivoUtil.generar_codigo('MUL'), cod_prestamo, dias_retraso,
                      dias_retraso * MULTA_POR_DIA, EstadoMultaEnum.PENDIENTE)
        self.multa_repo.guardar(multa)
        return multa

    def pagar(self, codigo):
        """Registra el pago de una multa pendiente."""
        multa = self.multa_repo.find_by_codigo(codigo)
        if multa is None:
            raise ValueError('La multa no existe.')
        if multa.estado == EstadoMultaEnum.PAGADA:
            raise ValueError('La multa ya fue pagada.')
        multa.estado = EstadoMultaEnum.PAGADA
        self.multa_repo.actualizar(multa)
        return multa

    def pendientes_usuario(self, doc):
        """Retorna las multas pendientes de un usuario."""
        return self.multa_repo.find_pendientes_usuario(doc)

    def tiene_pendientes(self, doc):
        """Indica si el usuario tiene al menos una multa pendiente."""
        return bool(self.multa_repo.find_pendientes_usuario(doc))

    def listar_todas(self):
        """Retorna todas las multas registradas."""
        return self.multa_repo.find_all()

### Celda 3.4: ReservaService

In [54]:
class ReservaService:
    """Reglas de negocio para reservas por orden de llegada (RF-04)."""

    def __init__(self, reserva_repo, libro_repo, usuario_repo):
        self.reserva_repo = reserva_repo
        self.libro_repo = libro_repo
        self.usuario_repo = usuario_repo

    def reservar(self, num_documento, isbn, fecha=None):
        """Reserva un libro sin ejemplares disponibles para un usuario activo."""
        usuario = self.usuario_repo.find_by_documento(num_documento)
        if usuario is None:
            raise ValueError('El usuario no existe.')
        if usuario.estado != ESTADO_USUARIO_ACTIVO:
            raise ValueError('El usuario no está activo.')
        libro = self.libro_repo.find_by_isbn(isbn)
        if libro is None:
            raise ValueError('El libro no existe.')
        if libro.disponibles > 0:
            raise ValueError('El libro tiene ejemplares disponibles; no necesita reserva.')
        for r in self.reserva_repo.find_activas_libro(isbn):
            if r.num_documento == num_documento:
                raise ValueError('El usuario ya tiene una reserva activa de este libro.')
        hoy = (fecha or date.today()).strftime(FORMATO_FECHA)
        reserva = Reserva(ArchivoUtil.generar_codigo('RES'), isbn, num_documento, hoy)
        self.reserva_repo.guardar(reserva)
        return reserva

    def cancelar(self, codigo):
        """Cancela una reserva activa."""
        reserva = self.reserva_repo.find_by_codigo(codigo)
        if reserva is None:
            raise ValueError('La reserva no existe.')
        if reserva.estado != EstadoReservaEnum.ACTIVA:
            raise ValueError('Solo se pueden cancelar reservas activas.')
        reserva.estado = EstadoReservaEnum.CANCELADA
        self.reserva_repo.actualizar(reserva)
        return reserva

    def cola_libro(self, isbn):
        """Retorna las reservas activas de un libro en orden de llegada."""
        return self.reserva_repo.find_activas_libro(isbn)

    def primera_activa(self, isbn):
        """Retorna la primera reserva activa de un libro, o None si no hay cola."""
        cola = self.reserva_repo.find_activas_libro(isbn)
        return cola[0] if cola else None

    def cumplir(self, reserva):
        """Marca una reserva como CUMPLIDA (el usuario ya prestó el libro)."""
        reserva.estado = EstadoReservaEnum.CUMPLIDA
        self.reserva_repo.actualizar(reserva)
        return reserva

    def listar_por_libro(self, isbn):
        """Retorna todas las reservas de un libro."""
        return self.reserva_repo.find_by_libro(isbn)

    def listar_por_usuario(self, doc):
        """Retorna todas las reservas de un usuario."""
        return self.reserva_repo.find_by_usuario(doc)

    def listar_todas(self):
        """Retorna todas las reservas registradas."""
        return self.reserva_repo.find_all()

### Celda 3.5: PrestamoService

In [55]:
class PrestamoService:
    """Reglas de negocio para préstamos y devoluciones (RF-03 y RF-04)."""

    def __init__(self, prestamo_repo, libro_repo, usuario_repo, multa_service, reserva_service):
        self.prestamo_repo = prestamo_repo
        self.libro_repo = libro_repo
        self.usuario_repo = usuario_repo
        self.multa_service = multa_service
        self.reserva_service = reserva_service

    def prestar(self, num_documento, isbn, fecha_prestamo=None):
        """Registra un préstamo validando usuario, multas, cupo, disponibilidad y turno."""
        hoy = fecha_prestamo or date.today()
        usuario = self.usuario_repo.find_by_documento(num_documento)
        if usuario is None:
            raise ValueError('El usuario no existe.')
        if usuario.estado != ESTADO_USUARIO_ACTIVO:
            raise ValueError('El usuario no está activo.')
        if self.multa_service.tiene_pendientes(num_documento):
            raise ValueError('El usuario tiene multas pendientes.')

        abiertos = [p for p in self.prestamo_repo.find_by_usuario(num_documento)
                    if p.estado != EstadoPrestamoEnum.DEVUELTO]
        if len(abiertos) >= CUPO_LIBROS[usuario.tipo]:
            raise ValueError(f'El usuario alcanzó su cupo de {CUPO_LIBROS[usuario.tipo]} libros.')

        libro = self.libro_repo.find_by_isbn(isbn)
        if libro is None:
            raise ValueError('El libro no existe.')
        if libro.disponibles < 1:
            raise ValueError('No hay ejemplares disponibles de este libro.')

        primera = self.reserva_service.primera_activa(isbn)
        if primera is not None and primera.num_documento != num_documento:
            raise ValueError('El libro está reservado y no es el turno de este usuario.')

        limite = hoy + timedelta(days=DIAS_PRESTAMO[usuario.tipo])
        prestamo = Prestamo(ArchivoUtil.generar_codigo('PRE'), isbn, num_documento,
                            hoy.strftime(FORMATO_FECHA), limite.strftime(FORMATO_FECHA))
        self.prestamo_repo.guardar(prestamo)
        libro.disponibles -= 1
        self.libro_repo.actualizar(libro)
        if primera is not None:
            self.reserva_service.cumplir(primera)
        return prestamo

    def devolver(self, codigo, fecha_devolucion=None):
        """Registra la devolución, actualiza el libro y genera la multa si hay retraso.

        Retorna (prestamo, dias_retraso, multa). multa es None si no hubo retraso.
        """
        hoy = fecha_devolucion or date.today()
        prestamo = self.prestamo_repo.find_by_codigo(codigo)
        if prestamo is None:
            raise ValueError('El préstamo no existe.')
        if prestamo.estado == EstadoPrestamoEnum.DEVUELTO:
            raise ValueError('El préstamo ya fue devuelto.')

        prestamo.fecha_devolucion = hoy.strftime(FORMATO_FECHA)
        prestamo.estado = EstadoPrestamoEnum.DEVUELTO
        self.prestamo_repo.actualizar(prestamo)

        libro = self.libro_repo.find_by_isbn(prestamo.isbn)
        if libro is not None and libro.disponibles < libro.total_ejemplares:
            libro.disponibles += 1
            self.libro_repo.actualizar(libro)

        limite = datetime.strptime(prestamo.fecha_limite, FORMATO_FECHA).date()
        dias_retraso = max(0, (hoy - limite).days)
        multa = None
        if dias_retraso > 0:
            multa = self.multa_service.generar(prestamo.codigo, dias_retraso)
        return prestamo, dias_retraso, multa

    def buscar_por_codigo(self, codigo):
        """Retorna el préstamo con ese código, o None."""
        return self.prestamo_repo.find_by_codigo(codigo)

    def historial_usuario(self, num_documento):
        """Retorna los préstamos de un usuario en orden cronológico."""
        return sorted(self.prestamo_repo.find_by_usuario(num_documento),
                      key=lambda p: p.fecha_prestamo)

    def listar_todos(self):
        """Retorna todos los préstamos registrados."""
        return self.prestamo_repo.find_all()

    def listar_vencidos(self, fecha=None):
        """Retorna los préstamos no devueltos cuya fecha límite es anterior a fecha."""
        ref = (fecha or date.today()).strftime(FORMATO_FECHA)
        return [p for p in self.prestamo_repo.find_all()
                if p.estado != EstadoPrestamoEnum.DEVUELTO and p.fecha_limite < ref]

    def marcar_vencidos(self, fecha=None):
        """Cambia a VENCIDO los préstamos ACTIVO con fecha límite anterior a fecha."""
        ref = (fecha or date.today()).strftime(FORMATO_FECHA)
        marcados = []
        for p in self.prestamo_repo.find_all():
            if p.estado == EstadoPrestamoEnum.ACTIVO and p.fecha_limite < ref:
                p.estado = EstadoPrestamoEnum.VENCIDO
                self.prestamo_repo.actualizar(p)
                marcados.append(p)
        return marcados

### Celda 3.6: AnalisisDatosService

In [56]:
from collections import Counter


class AnalisisDatosService:
    """Indicadores estadísticos de uso de la colección (RF-05)."""

    def __init__(self, prestamo_repo, libro_repo, usuario_repo, categoria_repo, multa_repo):
        self.prestamo_repo = prestamo_repo
        self.libro_repo = libro_repo
        self.usuario_repo = usuario_repo
        self.categoria_repo = categoria_repo
        self.multa_repo = multa_repo

    def libros_mas_prestados(self, desde=None, hasta=None, n=5):
        """Retorna los n libros más prestados como (isbn, título, cantidad).

        Cuenta por fecha de préstamo. Una fecha None deja el periodo abierto en ese extremo.
        """
        d = desde.strftime(FORMATO_FECHA) if desde else None
        h = hasta.strftime(FORMATO_FECHA) if hasta else None
        conteo = Counter()
        for p in self.prestamo_repo.find_all():
            if d and p.fecha_prestamo < d:
                continue
            if h and p.fecha_prestamo > h:
                continue
            conteo[p.isbn] += 1
        filas = []
        for isbn, cantidad in conteo.items():
            libro = self.libro_repo.find_by_isbn(isbn)
            filas.append((isbn, libro.titulo if libro else isbn, cantidad))
        filas.sort(key=lambda f: -f[2])
        return filas[:n]

    def prestamos_por_categoria(self):
        """Retorna (código, nombre, cantidad, porcentaje) de préstamos por categoría."""
        categoria_de = {l.isbn: l.cod_categoria for l in self.libro_repo.find_all()}
        conteo = Counter()
        for p in self.prestamo_repo.find_all():
            if p.isbn in categoria_de:
                conteo[categoria_de[p.isbn]] += 1
        total = sum(conteo.values())
        filas = []
        for cod, cantidad in conteo.items():
            cat = self.categoria_repo.find_by_codigo(cod)
            filas.append((cod, cat.nombre if cat else cod, cantidad,
                          round(cantidad * 100 / total, 1)))
        filas.sort(key=lambda f: -f[2])
        return filas

    @staticmethod
    def _es_moroso(p):
        """Un préstamo es moroso si está VENCIDO o se devolvió después de la fecha límite."""
        if p.estado == EstadoPrestamoEnum.VENCIDO:
            return True
        return p.fecha_devolucion != NINGUNO and p.fecha_devolucion > p.fecha_limite

    def morosidad_por_tipo(self):
        """Retorna (tipo, morosos, total, tasa %) por tipo de usuario."""
        tipo_de = {u.num_documento: u.tipo for u in self.usuario_repo.find_all()}
        total = {t: 0 for t in TipoUsuarioEnum}
        morosos = {t: 0 for t in TipoUsuarioEnum}
        for p in self.prestamo_repo.find_all():
            tipo = tipo_de.get(p.num_documento)
            if tipo is None:
                continue
            total[tipo] += 1
            if self._es_moroso(p):
                morosos[tipo] += 1
        return [(t.value, morosos[t], total[t],
                 round(morosos[t] * 100 / total[t], 1) if total[t] else 0.0)
                for t in TipoUsuarioEnum]

    def duracion_promedio_prestamo(self):
        """Retorna (promedio de días, cantidad) de los préstamos DEVUELTO."""
        dias = []
        for p in self.prestamo_repo.find_all():
            if p.estado == EstadoPrestamoEnum.DEVUELTO and p.fecha_devolucion != NINGUNO:
                ini = datetime.strptime(p.fecha_prestamo, FORMATO_FECHA).date()
                fin = datetime.strptime(p.fecha_devolucion, FORMATO_FECHA).date()
                dias.append((fin - ini).days)
        return (sum(dias) / len(dias) if dias else 0.0), len(dias)

    def reporte_multas(self):
        """Retorna (generado, pagado, pendiente) en COP."""
        generado = pagado = pendiente = 0
        for m in self.multa_repo.find_all():
            generado += m.valor
            if m.estado == EstadoMultaEnum.PAGADA:
                pagado += m.valor
            else:
                pendiente += m.valor
        return generado, pagado, pendiente

### Celda 3.7: crear_servicios (ensamblado)

In [57]:
import types


def crear_servicios(carpeta=None):
    """Crea los repositorios y servicios apuntando a una carpeta de datos.

    Sin argumento usa RUTA_BASE (Google Drive). Con una carpeta distinta
    (por ejemplo una temporal) las pruebas no tocan los datos reales.
    """
    base = carpeta or RUTA_BASE
    os.makedirs(base, exist_ok=True)

    def ruta(nombre):
        return os.path.join(base, nombre)

    usuario_repo = UsuarioRepository(ruta('usuarios.txt'))
    categoria_repo = CategoriaRepository(ruta('categorias.txt'))
    libro_repo = LibroRepository(ruta('libros.txt'))
    prestamo_repo = PrestamoRepository(ruta('prestamos.txt'))
    multa_repo = MultaRepository(ruta('multas.txt'), prestamo_repo)
    reserva_repo = ReservaRepository(ruta('reservas.txt'))

    multa_s = MultaService(multa_repo, prestamo_repo)
    reserva_s = ReservaService(reserva_repo, libro_repo, usuario_repo)
    return types.SimpleNamespace(
        usuario=UsuarioService(usuario_repo),
        libro=LibroService(libro_repo, categoria_repo),
        multa=multa_s,
        reserva=reserva_s,
        prestamo=PrestamoService(prestamo_repo, libro_repo, usuario_repo, multa_s, reserva_s),
        analisis=AnalisisDatosService(prestamo_repo, libro_repo, usuario_repo,
                                      categoria_repo, multa_repo),
    )

## Sección 4: interfaz de usuario (Gradio)

### Celda 4.0: servicios y funciones auxiliares

In [58]:
# Servicios que usa la interfaz (trabajan sobre los archivos de RUTA_BASE)
S = crear_servicios()
usuario_service = S.usuario
libro_service = S.libro
prestamo_service = S.prestamo
reserva_service = S.reserva
multa_service = S.multa
analisis_service = S.analisis

H_USUARIOS = ['Documento', 'Nombre', 'Tipo', 'Correo', 'Teléfono', 'Estado']
H_CATEGORIAS = ['Código', 'Nombre', 'Descripción']
H_LIBROS = ['ISBN', 'Título', 'Autor', 'Categoría', 'Año', 'Disponibles', 'Total']
H_PRESTAMOS = ['Código', 'ISBN', 'Documento', 'Fecha préstamo', 'Fecha límite',
               'Fecha devolución', 'Estado']
H_RESERVAS = ['Código', 'ISBN', 'Documento', 'Fecha reserva', 'Estado']
H_MULTAS = ['Código', 'Préstamo', 'Días de retraso', 'Valor (COP)', 'Estado']


def _tabla(filas, columnas):
    """Retorna las filas, o una fila vacía si no hay resultados."""
    return filas if filas else [[''] * len(columnas)]


def _cop(valor):
    """Formatea un valor en pesos con punto de miles (COP 11.000)."""
    return 'COP ' + f'{valor:,}'.replace(',', '.')


def usuarios_a_filas(usuarios):
    """Convierte una lista de Usuario en filas para la tabla."""
    return [[u.num_documento, u.nombre, u.tipo.value, u.correo, u.telefono, u.estado]
            for u in usuarios]


def categorias_a_filas(categorias):
    """Convierte una lista de Categoria en filas para la tabla."""
    return [[c.codigo, c.nombre, c.descripcion] for c in categorias]


def libros_a_filas(libros):
    """Convierte una lista de Libro en filas para la tabla."""
    return [[l.isbn, l.titulo, l.autor, l.cod_categoria, l.anio,
             l.disponibles, l.total_ejemplares] for l in libros]


def prestamos_a_filas(prestamos):
    """Convierte una lista de Prestamo en filas para la tabla."""
    return [[p.codigo, p.isbn, p.num_documento, p.fecha_prestamo, p.fecha_limite,
             p.fecha_devolucion, p.estado.value] for p in prestamos]


def reservas_a_filas(reservas):
    """Convierte una lista de Reserva en filas para la tabla."""
    return [[r.codigo, r.isbn, r.num_documento, r.fecha_reserva, r.estado.value]
            for r in reservas]


def multas_a_filas(multas):
    """Convierte una lista de Multa en filas para la tabla."""
    return [[m.codigo, m.cod_prestamo, m.dias_retraso, m.valor, m.estado.value]
            for m in multas]


def _texto(valor, nombre):
    """Valida un texto obligatorio que no contenga el carácter |."""
    v = (valor or '').strip()
    if not v:
        raise ValueError(f'El campo {nombre} es obligatorio.')
    if '|' in v:
        raise ValueError(f'El campo {nombre} no puede contener el carácter |.')
    return v


def _texto_opcional(valor, nombre):
    """Valida un texto opcional que no contenga el carácter |."""
    v = (valor or '').strip()
    if '|' in v:
        raise ValueError(f'El campo {nombre} no puede contener el carácter |.')
    return v


def _entero(valor, nombre, minimo=None):
    """Convierte un valor numérico de la interfaz en entero."""
    if valor is None or valor == '':
        raise ValueError(f'El campo {nombre} es obligatorio.')
    try:
        n = int(valor)
    except (TypeError, ValueError):
        raise ValueError(f'El campo {nombre} debe ser un número entero.')
    if minimo is not None and n < minimo:
        raise ValueError(f'El campo {nombre} debe ser al menos {minimo}.')
    return n


def _fecha(valor, nombre):
    """Convierte AAAA-MM-DD a date. Un texto vacío retorna None (se usa hoy)."""
    v = (valor or '').strip()
    if not v:
        return None
    try:
        return datetime.strptime(v, FORMATO_FECHA).date()
    except ValueError:
        raise ValueError(f'{nombre} inválida. Use el formato AAAA-MM-DD.')

### Celda 4.1: funciones de la pestaña usuarios

In [59]:
def _tabla_usuarios():
    """Tabla con todos los usuarios registrados."""
    return _tabla(usuarios_a_filas(usuario_service.listar_todos()), H_USUARIOS)


def ui_registrar_usuario(doc, nombre, tipo, correo, telefono):
    """Registra un usuario desde el formulario."""
    try:
        u = usuario_service.registrar(_texto(doc, 'documento'), _texto(nombre, 'nombre'),
                                      TipoUsuarioEnum(tipo), _texto(correo, 'correo'),
                                      _texto(telefono, 'teléfono'))
        return f'Usuario registrado: {u.nombre} ({u.num_documento}).', _tabla_usuarios()
    except ValueError as e:
        return f'Error: {e}', _tabla_usuarios()


def ui_consultar_usuario(doc):
    """Consulta un usuario por documento."""
    try:
        u = usuario_service.buscar_por_documento(_texto(doc, 'documento'))
    except ValueError as e:
        return f'Error: {e}', _tabla_usuarios()
    if u is None:
        return 'No se encontró el usuario.', _tabla([], H_USUARIOS)
    return 'Usuario encontrado.', usuarios_a_filas([u])


def ui_actualizar_usuario(doc, correo, telefono, estado):
    """Actualiza correo, teléfono o estado de un usuario."""
    try:
        estado_nuevo = None if estado == 'Sin cambio' else estado
        u = usuario_service.actualizar(_texto(doc, 'documento'),
                                       _texto_opcional(correo, 'correo'),
                                       _texto_opcional(telefono, 'teléfono'),
                                       estado_nuevo)
        return f'Usuario actualizado: {u.nombre}.', usuarios_a_filas([u])
    except ValueError as e:
        return f'Error: {e}', _tabla_usuarios()


def ui_listar_usuarios():
    """Lista todos los usuarios."""
    return f'{len(usuario_service.listar_todos())} usuario(s) registrado(s).', _tabla_usuarios()

### Celda 4.2: funciones de la pestaña libros y categorías

In [60]:
def _tabla_libros():
    """Tabla con todos los libros del catálogo."""
    return _tabla(libros_a_filas(libro_service.listar_todos()), H_LIBROS)


def _tabla_categorias():
    """Tabla con todas las categorías."""
    return _tabla(categorias_a_filas(libro_service.listar_categorias()), H_CATEGORIAS)


def ui_registrar_categoria(codigo, nombre, descripcion):
    """Registra una categoría desde el formulario."""
    try:
        c = libro_service.registrar_categoria(_texto(codigo, 'código'), _texto(nombre, 'nombre'),
                                              _texto(descripcion, 'descripción'))
        return f'Categoría registrada: {c.codigo}.', _tabla_categorias()
    except ValueError as e:
        return f'Error: {e}', _tabla_categorias()


def ui_registrar_libro(isbn, titulo, autor, cod_categoria, anio, total):
    """Registra un libro desde el formulario."""
    try:
        anio_n = _entero(anio, 'año de publicación', 0)
        if anio_n > date.today().year:
            raise ValueError('El año de publicación no puede ser futuro.')
        l = libro_service.registrar(_texto(isbn, 'ISBN'), _texto(titulo, 'título'),
                                    _texto(autor, 'autor'), _texto(cod_categoria, 'categoría'),
                                    anio_n, _entero(total, 'ejemplares', 1))
        return f'Libro registrado: {l.titulo}.', _tabla_libros()
    except ValueError as e:
        return f'Error: {e}', _tabla_libros()


def ui_buscar_isbn(texto):
    """Busca un libro por ISBN."""
    try:
        l = libro_service.buscar_por_isbn(_texto(texto, 'búsqueda'))
    except ValueError as e:
        return f'Error: {e}', _tabla_libros()
    if l is None:
        return 'No se encontró el libro.', _tabla([], H_LIBROS)
    return 'Libro encontrado.', libros_a_filas([l])


def ui_buscar_titulo(texto):
    """Busca libros cuyo título contiene el texto."""
    try:
        res = libro_service.buscar_por_titulo(_texto(texto, 'búsqueda'))
    except ValueError as e:
        return f'Error: {e}', _tabla_libros()
    return f'{len(res)} resultado(s).', _tabla(libros_a_filas(res), H_LIBROS)


def ui_buscar_categoria(texto):
    """Lista los libros de una categoría."""
    try:
        res = libro_service.listar_por_categoria(_texto(texto, 'búsqueda'))
    except ValueError as e:
        return f'Error: {e}', _tabla_libros()
    return f'{len(res)} resultado(s).', _tabla(libros_a_filas(res), H_LIBROS)


def ui_listar_disponibles():
    """Lista los libros con ejemplares disponibles."""
    res = libro_service.listar_disponibles()
    return f'{len(res)} libro(s) con ejemplares disponibles.', _tabla(libros_a_filas(res), H_LIBROS)


def ui_listar_libros():
    """Lista todos los libros del catálogo."""
    return f'{len(libro_service.listar_todos())} libro(s) en el catálogo.', _tabla_libros()


def ui_listar_categorias():
    """Lista todas las categorías."""
    return f'{len(libro_service.listar_categorias())} categoría(s).', _tabla_categorias()

### Celda 4.3: funciones de la pestaña préstamos

In [61]:
def ui_prestar(doc, isbn, fecha):
    """Registra un préstamo desde el formulario."""
    try:
        p = prestamo_service.prestar(_texto(doc, 'documento'), _texto(isbn, 'ISBN'),
                                     _fecha(fecha, 'Fecha de préstamo'))
        return (f'Préstamo registrado: {p.codigo}. Fecha límite: {p.fecha_limite}.',
                prestamos_a_filas([p]))
    except ValueError as e:
        return f'Error: {e}', _tabla([], H_PRESTAMOS)


def ui_devolver(codigo, fecha):
    """Registra la devolución de un préstamo e informa la multa si hubo retraso."""
    try:
        p, retraso, multa = prestamo_service.devolver(_texto(codigo, 'código'),
                                                      _fecha(fecha, 'Fecha de devolución'))
        msg = f'Devolución registrada: {p.codigo}.'
        if retraso > 0:
            msg += (f' Devuelto con {retraso} día(s) de retraso. '
                    f'Multa generada: {multa.codigo} por {_cop(multa.valor)}.')
        return msg, prestamos_a_filas([p])
    except ValueError as e:
        return f'Error: {e}', _tabla([], H_PRESTAMOS)


def ui_consultar_prestamo(codigo):
    """Consulta un préstamo por código."""
    try:
        p = prestamo_service.buscar_por_codigo(_texto(codigo, 'código'))
    except ValueError as e:
        return f'Error: {e}', _tabla([], H_PRESTAMOS)
    if p is None:
        return 'No se encontró el préstamo.', _tabla([], H_PRESTAMOS)
    return 'Préstamo encontrado.', prestamos_a_filas([p])


def ui_historial(doc):
    """Lista los préstamos de un usuario en orden cronológico."""
    try:
        res = prestamo_service.historial_usuario(_texto(doc, 'documento'))
    except ValueError as e:
        return f'Error: {e}', _tabla([], H_PRESTAMOS)
    return f'{len(res)} préstamo(s) del usuario.', _tabla(prestamos_a_filas(res), H_PRESTAMOS)


def ui_vencidos(fecha):
    """Lista los préstamos vencidos a una fecha de corte."""
    try:
        res = prestamo_service.listar_vencidos(_fecha(fecha, 'Fecha de corte'))
    except ValueError as e:
        return f'Error: {e}', _tabla([], H_PRESTAMOS)
    return f'{len(res)} préstamo(s) vencido(s).', _tabla(prestamos_a_filas(res), H_PRESTAMOS)


def ui_listar_prestamos():
    """Lista todos los préstamos registrados."""
    res = prestamo_service.listar_todos()
    return f'{len(res)} préstamo(s) registrado(s).', _tabla(prestamos_a_filas(res), H_PRESTAMOS)

### Celda 4.4: funciones de la pestaña reservas y multas

In [62]:
def _tabla_reservas():
    """Tabla con todas las reservas."""
    return _tabla(reservas_a_filas(reserva_service.listar_todas()), H_RESERVAS)


def _tabla_multas():
    """Tabla con todas las multas."""
    return _tabla(multas_a_filas(multa_service.listar_todas()), H_MULTAS)


def ui_reservar(doc, isbn):
    """Reserva un libro sin ejemplares y muestra la cola del libro."""
    try:
        r = reserva_service.reservar(_texto(doc, 'documento'), _texto(isbn, 'ISBN'))
        cola = reserva_service.cola_libro(r.isbn)
        posicion = [x.codigo for x in cola].index(r.codigo) + 1
        return (f'Reserva registrada: {r.codigo}. Posición en la cola: {posicion}.',
                reservas_a_filas(cola))
    except ValueError as e:
        return f'Error: {e}', _tabla_reservas()


def ui_cancelar_reserva(codigo):
    """Cancela una reserva activa."""
    try:
        r = reserva_service.cancelar(_texto(codigo, 'código de reserva'))
        return f'Reserva cancelada: {r.codigo}.', _tabla_reservas()
    except ValueError as e:
        return f'Error: {e}', _tabla_reservas()


def ui_reservas_libro(isbn):
    """Lista las reservas de un libro en orden de llegada."""
    try:
        res = reserva_service.listar_por_libro(_texto(isbn, 'ISBN'))
    except ValueError as e:
        return f'Error: {e}', _tabla_reservas()
    return f'{len(res)} reserva(s) del libro.', _tabla(reservas_a_filas(res), H_RESERVAS)


def ui_reservas_usuario(doc):
    """Lista las reservas de un usuario."""
    try:
        res = reserva_service.listar_por_usuario(_texto(doc, 'documento'))
    except ValueError as e:
        return f'Error: {e}', _tabla_reservas()
    return f'{len(res)} reserva(s) del usuario.', _tabla(reservas_a_filas(res), H_RESERVAS)


def ui_listar_reservas():
    """Lista todas las reservas."""
    return f'{len(reserva_service.listar_todas())} reserva(s) registrada(s).', _tabla_reservas()


def ui_multas_pendientes(doc):
    """Lista las multas pendientes de un usuario."""
    try:
        res = multa_service.pendientes_usuario(_texto(doc, 'documento'))
    except ValueError as e:
        return f'Error: {e}', _tabla_multas()
    total = sum(m.valor for m in res)
    return (f'{len(res)} multa(s) pendiente(s) por {_cop(total)}.',
            _tabla(multas_a_filas(res), H_MULTAS))


def ui_pagar_multa(codigo):
    """Registra el pago de una multa."""
    try:
        m = multa_service.pagar(_texto(codigo, 'código de multa'))
        return f'Multa pagada: {m.codigo} por {_cop(m.valor)}.', _tabla_multas()
    except ValueError as e:
        return f'Error: {e}', _tabla_multas()


def ui_listar_multas():
    """Lista todas las multas."""
    return f'{len(multa_service.listar_todas())} multa(s) registrada(s).', _tabla_multas()

### Celda 4.5: funciones de la pestaña análisis de datos

In [63]:
import pandas as pd   # Gradio ya lo instala; solo la interfaz lo usa


def _df(filas, columnas):
    """Construye la tabla de resultados del análisis."""
    return pd.DataFrame(filas, columns=columnas)


def ui_top_libros(desde, hasta, n):
    """Indicador: libros más prestados en un periodo."""
    try:
        res = analisis_service.libros_mas_prestados(_fecha(desde, 'Fecha desde'),
                                                    _fecha(hasta, 'Fecha hasta'),
                                                    _entero(n, 'N', 1))
    except ValueError as e:
        return f'Error: {e}', _df([], ['Resultado'])
    filas = [[i + 1, isbn, titulo, cant] for i, (isbn, titulo, cant) in enumerate(res)]
    return f'Los {len(res)} libros más prestados del periodo.', _df(filas, ['Puesto', 'ISBN', 'Título', 'Préstamos'])


def ui_por_categoria():
    """Indicador: distribución de préstamos por categoría."""
    res = analisis_service.prestamos_por_categoria()
    filas = [[cod, nombre, cant, f'{pct:.1f} %'] for cod, nombre, cant, pct in res]
    total = sum(f[2] for f in res)
    return f'{total} préstamo(s) en total.', _df(filas, ['Código', 'Categoría', 'Préstamos', 'Porcentaje'])


def ui_morosidad():
    """Indicador: tasa de morosidad por tipo de usuario (marca antes los vencidos)."""
    marcados = prestamo_service.marcar_vencidos()
    res = analisis_service.morosidad_por_tipo()
    filas = [[tipo, mor, tot, f'{tasa:.1f} %'] for tipo, mor, tot, tasa in res]
    return (f'Se marcaron {len(marcados)} préstamo(s) como VENCIDO antes de calcular.',
            _df(filas, ['Tipo de usuario', 'Préstamos morosos', 'Total préstamos', 'Tasa de morosidad']))


def ui_duracion():
    """Indicador: duración promedio real de los préstamos devueltos."""
    prom, cant = analisis_service.duracion_promedio_prestamo()
    return (f'Duración promedio: {prom:.1f} días.',
            _df([[f'{prom:.1f}', cant]], ['Duración promedio (días)', 'Préstamos devueltos']))


def ui_reporte_multas():
    """Indicador: valor de multas generado, pagado y pendiente."""
    gen, pag, pen = analisis_service.reporte_multas()
    return ('Reporte de multas.',
            _df([[_cop(gen), _cop(pag), _cop(pen)]], ['Generado', 'Pagado', 'Pendiente']))

### Celda 4.6: funciones de la pestaña administración

In [64]:
def ui_cargar_datos():
    """Carga los datos de prueba (reemplazan los actuales) y refresca las tablas."""
    r = cargar_datos_prueba()
    msg = (f"Datos de prueba cargados: {r['categorias']} categorías, {r['usuarios']} usuarios, "
           f"{r['libros']} libros, {r['prestamos']} préstamos, {r['multas']} multas y "
           f"{r['reservas']} reservas. Reemplazaron los datos anteriores.")
    return (msg, _tabla_usuarios(), _tabla_libros(), _tabla_categorias(),
            _tabla(prestamos_a_filas(prestamo_service.listar_todos()), H_PRESTAMOS),
            _tabla_reservas(), _tabla_multas())


def ui_marcar_vencidos():
    """Marca como VENCIDO los préstamos activos con fecha límite anterior a hoy."""
    marcados = prestamo_service.marcar_vencidos()
    return (f'{len(marcados)} préstamo(s) marcado(s) como VENCIDO.',
            _tabla(prestamos_a_filas(marcados), H_PRESTAMOS))


def ui_ejecutar_pruebas():
    """Ejecuta las pruebas de integración (sobre una carpeta temporal) y muestra el informe."""
    ok, informe = ejecutar_pruebas()
    return ('Todas las pruebas pasaron.' if ok else 'Hay pruebas que fallaron.'), informe

### Celda 4.7: construcción de la interfaz

In [65]:
def construir_interfaz():
    """Arma la interfaz de Gradio con una pestaña por módulo."""
    with gr.Blocks(title='Sistema de Gestión de Biblioteca') as demo:
        gr.Markdown('# Sistema de Gestión de Biblioteca')

        # ---------------- usuarios ----------------
        with gr.Tab('Usuarios'):
            with gr.Row():
                with gr.Column():
                    gr.Markdown('### Registrar usuario')
                    u_doc = gr.Textbox(label='Número de documento')
                    u_nombre = gr.Textbox(label='Nombre completo')
                    u_tipo = gr.Dropdown(choices=[t.value for t in TipoUsuarioEnum],
                                         value=TipoUsuarioEnum.ESTUDIANTE.value,
                                         label='Tipo de usuario')
                    u_correo = gr.Textbox(label='Correo')
                    u_tel = gr.Textbox(label='Teléfono')
                    u_reg = gr.Button('Registrar', variant='primary')
                with gr.Column():
                    gr.Markdown('### Consultar o actualizar')
                    u_doc2 = gr.Textbox(label='Documento del usuario')
                    u_correo2 = gr.Textbox(label='Nuevo correo (vacío para no cambiar)')
                    u_tel2 = gr.Textbox(label='Nuevo teléfono (vacío para no cambiar)')
                    u_estado = gr.Dropdown(choices=['Sin cambio', ESTADO_USUARIO_ACTIVO,
                                                    ESTADO_USUARIO_SUSPENDIDO],
                                           value='Sin cambio', label='Estado')
                    with gr.Row():
                        u_cons = gr.Button('Consultar')
                        u_act = gr.Button('Actualizar', variant='primary')
                    u_list = gr.Button('Listar todos')
            u_msg = gr.Markdown()
            u_tabla = gr.Dataframe(headers=H_USUARIOS, value=_tabla_usuarios(),
                                   interactive=False, label='Usuarios')
            u_reg.click(ui_registrar_usuario, [u_doc, u_nombre, u_tipo, u_correo, u_tel],
                        [u_msg, u_tabla])
            u_cons.click(ui_consultar_usuario, [u_doc2], [u_msg, u_tabla])
            u_act.click(ui_actualizar_usuario, [u_doc2, u_correo2, u_tel2, u_estado],
                        [u_msg, u_tabla])
            u_list.click(ui_listar_usuarios, None, [u_msg, u_tabla])

        # ---------------- libros y categorías ----------------
        with gr.Tab('Libros y categorías'):
            with gr.Row():
                with gr.Column():
                    gr.Markdown('### Registrar categoría')
                    c_cod = gr.Textbox(label='Código de categoría')
                    c_nom = gr.Textbox(label='Nombre')
                    c_desc = gr.Textbox(label='Descripción')
                    c_reg = gr.Button('Registrar categoría', variant='primary')
                    c_list = gr.Button('Listar categorías')
                with gr.Column():
                    gr.Markdown('### Registrar libro')
                    l_isbn = gr.Textbox(label='ISBN')
                    l_titulo = gr.Textbox(label='Título')
                    l_autor = gr.Textbox(label='Autor')
                    l_cat = gr.Textbox(label='Código de categoría')
                    l_anio = gr.Number(label='Año de publicación', precision=0)
                    l_total = gr.Number(label='Número de ejemplares', value=1, precision=0)
                    l_reg = gr.Button('Registrar libro', variant='primary')
            gr.Markdown('### Consultar libros')
            l_busca = gr.Textbox(label='Texto de búsqueda (ISBN, título o código de categoría)')
            with gr.Row():
                l_b_isbn = gr.Button('Buscar por ISBN')
                l_b_tit = gr.Button('Buscar por título')
                l_b_cat = gr.Button('Buscar por categoría')
                l_disp = gr.Button('Listar con ejemplares disponibles')
                l_todos = gr.Button('Listar todos')
            l_msg = gr.Markdown()
            l_tabla = gr.Dataframe(headers=H_LIBROS, value=_tabla_libros(),
                                   interactive=False, label='Libros')
            cat_tabla = gr.Dataframe(headers=H_CATEGORIAS, value=_tabla_categorias(),
                                     interactive=False, label='Categorías')
            c_reg.click(ui_registrar_categoria, [c_cod, c_nom, c_desc], [l_msg, cat_tabla])
            c_list.click(ui_listar_categorias, None, [l_msg, cat_tabla])
            l_reg.click(ui_registrar_libro, [l_isbn, l_titulo, l_autor, l_cat, l_anio, l_total],
                        [l_msg, l_tabla])
            l_b_isbn.click(ui_buscar_isbn, [l_busca], [l_msg, l_tabla])
            l_b_tit.click(ui_buscar_titulo, [l_busca], [l_msg, l_tabla])
            l_b_cat.click(ui_buscar_categoria, [l_busca], [l_msg, l_tabla])
            l_disp.click(ui_listar_disponibles, None, [l_msg, l_tabla])
            l_todos.click(ui_listar_libros, None, [l_msg, l_tabla])

        # ---------------- préstamos ----------------
        with gr.Tab('Préstamos'):
            with gr.Row():
                with gr.Column():
                    gr.Markdown('### Prestar libro')
                    p_doc = gr.Textbox(label='Documento del usuario')
                    p_isbn = gr.Textbox(label='ISBN del libro')
                    p_fecha = gr.Textbox(label='Fecha del préstamo (AAAA-MM-DD, vacío = hoy)')
                    p_prestar = gr.Button('Prestar', variant='primary')
                with gr.Column():
                    gr.Markdown('### Devolver libro')
                    d_cod = gr.Textbox(label='Código del préstamo')
                    d_fecha = gr.Textbox(label='Fecha de devolución (AAAA-MM-DD, vacío = hoy)')
                    d_devolver = gr.Button('Devolver', variant='primary')
            gr.Markdown('### Consultas')
            with gr.Row():
                q_cod = gr.Textbox(label='Código del préstamo')
                q_doc = gr.Textbox(label='Documento del usuario')
                q_corte = gr.Textbox(label='Fecha de corte (AAAA-MM-DD, vacío = hoy)')
            with gr.Row():
                q_b_cod = gr.Button('Consultar por código')
                q_b_hist = gr.Button('Historial del usuario')
                q_b_venc = gr.Button('Listar vencidos')
                q_b_todos = gr.Button('Listar todos')
            p_msg = gr.Markdown()
            p_tabla = gr.Dataframe(headers=H_PRESTAMOS, value=_tabla([], H_PRESTAMOS),
                                   interactive=False, label='Préstamos')
            p_prestar.click(ui_prestar, [p_doc, p_isbn, p_fecha], [p_msg, p_tabla])
            d_devolver.click(ui_devolver, [d_cod, d_fecha], [p_msg, p_tabla])
            q_b_cod.click(ui_consultar_prestamo, [q_cod], [p_msg, p_tabla])
            q_b_hist.click(ui_historial, [q_doc], [p_msg, p_tabla])
            q_b_venc.click(ui_vencidos, [q_corte], [p_msg, p_tabla])
            q_b_todos.click(ui_listar_prestamos, None, [p_msg, p_tabla])

        # ---------------- reservas y multas ----------------
        with gr.Tab('Reservas y multas'):
            with gr.Row():
                with gr.Column():
                    gr.Markdown('### Reservar libro')
                    r_doc = gr.Textbox(label='Documento del usuario')
                    r_isbn = gr.Textbox(label='ISBN del libro')
                    r_reservar = gr.Button('Reservar', variant='primary')
                    gr.Markdown('### Cancelar reserva')
                    r_cod = gr.Textbox(label='Código de la reserva')
                    r_cancelar = gr.Button('Cancelar reserva')
                with gr.Column():
                    gr.Markdown('### Multas')
                    m_doc = gr.Textbox(label='Documento del usuario')
                    m_pend = gr.Button('Ver multas pendientes')
                    m_cod = gr.Textbox(label='Código de la multa')
                    m_pagar = gr.Button('Pagar multa', variant='primary')
                    m_todas = gr.Button('Listar todas las multas')
            gr.Markdown('### Consultar reservas')
            with gr.Row():
                rq_isbn = gr.Textbox(label='ISBN del libro')
                rq_doc = gr.Textbox(label='Documento del usuario')
            with gr.Row():
                rq_b_libro = gr.Button('Reservas del libro')
                rq_b_usuario = gr.Button('Reservas del usuario')
                rq_b_todas = gr.Button('Listar todas las reservas')
            rm_msg = gr.Markdown()
            r_tabla = gr.Dataframe(headers=H_RESERVAS, value=_tabla_reservas(),
                                   interactive=False, label='Reservas')
            m_tabla = gr.Dataframe(headers=H_MULTAS, value=_tabla_multas(),
                                   interactive=False, label='Multas')
            r_reservar.click(ui_reservar, [r_doc, r_isbn], [rm_msg, r_tabla])
            r_cancelar.click(ui_cancelar_reserva, [r_cod], [rm_msg, r_tabla])
            rq_b_libro.click(ui_reservas_libro, [rq_isbn], [rm_msg, r_tabla])
            rq_b_usuario.click(ui_reservas_usuario, [rq_doc], [rm_msg, r_tabla])
            rq_b_todas.click(ui_listar_reservas, None, [rm_msg, r_tabla])
            m_pend.click(ui_multas_pendientes, [m_doc], [rm_msg, m_tabla])
            m_pagar.click(ui_pagar_multa, [m_cod], [rm_msg, m_tabla])
            m_todas.click(ui_listar_multas, None, [rm_msg, m_tabla])

        # ---------------- análisis de datos ----------------
        with gr.Tab('Análisis de datos'):
            gr.Markdown('### Libros más prestados (periodo opcional)')
            with gr.Row():
                a_desde = gr.Textbox(label='Desde (AAAA-MM-DD, vacío = sin límite)')
                a_hasta = gr.Textbox(label='Hasta (AAAA-MM-DD, vacío = sin límite)')
                a_n = gr.Number(label='Cantidad de libros (N)', value=5, precision=0)
            with gr.Row():
                a_b_top = gr.Button('Libros más prestados', variant='primary')
                a_b_cat = gr.Button('Préstamos por categoría')
                a_b_mor = gr.Button('Morosidad por tipo de usuario')
                a_b_dur = gr.Button('Duración promedio')
                a_b_mul = gr.Button('Reporte de multas')
            a_msg = gr.Markdown()
            a_tabla = gr.Dataframe(interactive=False, label='Resultado')
            a_b_top.click(ui_top_libros, [a_desde, a_hasta, a_n], [a_msg, a_tabla])
            a_b_cat.click(ui_por_categoria, None, [a_msg, a_tabla])
            a_b_mor.click(ui_morosidad, None, [a_msg, a_tabla])
            a_b_dur.click(ui_duracion, None, [a_msg, a_tabla])
            a_b_mul.click(ui_reporte_multas, None, [a_msg, a_tabla])

        # ---------------- administración ----------------
        with gr.Tab('Administración'):
            gr.Markdown('Cargar los datos de prueba reemplaza los archivos actuales. '
                        'Las pruebas de integración usan una carpeta temporal y no tocan los datos.')
            with gr.Row():
                adm_cargar = gr.Button('Cargar datos de prueba', variant='primary')
                adm_venc = gr.Button('Marcar préstamos vencidos')
                adm_pruebas = gr.Button('Ejecutar pruebas de integración')
            adm_msg = gr.Markdown()
            adm_tabla = gr.Dataframe(headers=H_PRESTAMOS, value=_tabla([], H_PRESTAMOS),
                                     interactive=False, label='Préstamos marcados')
            adm_log = gr.Textbox(label='Informe de pruebas', lines=16, interactive=False)
            adm_cargar.click(ui_cargar_datos, None,
                             [adm_msg, u_tabla, l_tabla, cat_tabla, p_tabla, r_tabla, m_tabla])
            adm_venc.click(ui_marcar_vencidos, None, [adm_msg, adm_tabla])
            adm_pruebas.click(ui_ejecutar_pruebas, None, [adm_msg, adm_log])
    return demo

## Sección 5: ejecución principal

### Celda 5.1: datos de prueba
Datos ficticios: 4 categorías, 6 usuarios, 6 libros, 12 préstamos, 3 multas y 2 reservas. Cubren los tres tipos de usuario, devoluciones a tiempo y con retraso, multas pagadas y pendientes, préstamos activos, un usuario suspendido y una reserva en cola.

In [66]:
ISBN_CIEN = '9780307474728'    # Cien años de soledad
ISBN_EST = '9789580000011'     # Estructuras de datos en Python
ISBN_BD = '9789580000028'      # Fundamentos de bases de datos
ISBN_ALG = '9789580000035'     # Álgebra lineal básica
ISBN_HIS = '9789580000042'     # Historia de Colombia
ISBN_RED = '9789580000059'     # Redes de computadores

DATOS_PRUEBA = {
    'categorias.txt': [
        'LIT|Literatura|Novelas, cuentos y poesía',
        'TEC|Tecnología|Programación, bases de datos y redes',
        'CIE|Ciencias|Matemáticas y ciencias básicas',
        'HIS|Historia|Historia de Colombia y del mundo',
    ],
    'usuarios.txt': [
        '1001|Maria Lopez|ESTUDIANTE|maria.lopez@correo.com|3001112233|ACTIVO',
        '1002|Juan Perez|DOCENTE|juan.perez@correo.com|3004445566|ACTIVO',
        '1003|Andres Torres|ESTUDIANTE|andres.torres@correo.com|3007778899|ACTIVO',
        '1004|Laura Rojas|DOCENTE|laura.rojas@correo.com|3101112233|ACTIVO',
        '1005|Pedro Salazar|EXTERNO|pedro.salazar@correo.com|3104445566|ACTIVO',
        '1006|Sofia Mendez|EXTERNO|sofia.mendez@correo.com|3107778899|SUSPENDIDO',
    ],
    'libros.txt': [
        f'{ISBN_CIEN}|Cien años de soledad|Gabriel García Márquez|LIT|1967|2|1',
        f'{ISBN_EST}|Estructuras de datos en Python|Ana Ruiz|TEC|2020|2|2',
        f'{ISBN_BD}|Fundamentos de bases de datos|Carlos Mejía|TEC|2019|1|0',
        f'{ISBN_ALG}|Álgebra lineal básica|Elena Vargas|CIE|2015|1|0',
        f'{ISBN_HIS}|Historia de Colombia|Jorge Orozco|HIS|2012|1|1',
        f'{ISBN_RED}|Redes de computadores|Marta Peña|TEC|2018|2|1',
    ],
    'prestamos.txt': [
        f'PRE-001|{ISBN_EST}|1001|2025-09-01|2025-09-08|2025-09-06|DEVUELTO',
        f'PRE-002|{ISBN_CIEN}|1001|2025-10-15|2025-10-22|2025-10-25|DEVUELTO',
        f'PRE-003|{ISBN_CIEN}|1003|2025-09-10|2025-09-17|2025-09-22|DEVUELTO',
        f'PRE-004|{ISBN_CIEN}|1003|2025-10-01|2025-10-08|2025-10-05|DEVUELTO',
        f'PRE-005|{ISBN_EST}|1003|2025-11-05|2025-11-12|2025-11-10|DEVUELTO',
        f'PRE-006|{ISBN_BD}|1003|2025-11-03|2025-11-10|NINGUNO|ACTIVO',
        f'PRE-007|{ISBN_CIEN}|1002|2025-11-01|2025-11-16|NINGUNO|ACTIVO',
        f'PRE-008|{ISBN_EST}|1002|2025-09-20|2025-10-05|2025-09-27|DEVUELTO',
        f'PRE-009|{ISBN_CIEN}|1004|2025-10-05|2025-10-20|2025-10-23|DEVUELTO',
        f'PRE-010|{ISBN_ALG}|1005|2025-11-10|2025-11-15|NINGUNO|ACTIVO',
        f'PRE-011|{ISBN_HIS}|1004|2025-10-25|2025-11-09|2025-11-03|DEVUELTO',
        f'PRE-012|{ISBN_RED}|1005|2025-11-12|2025-11-17|NINGUNO|ACTIVO',
    ],
    'multas.txt': [
        'MUL-001|PRE-002|3|3000|PENDIENTE',
        'MUL-002|PRE-003|5|5000|PAGADA',
        'MUL-003|PRE-009|3|3000|PAGADA',
    ],
    'reservas.txt': [
        f'RES-001|{ISBN_BD}|1004|2025-11-12|ACTIVA',
        f'RES-002|{ISBN_CIEN}|1001|2025-10-10|CUMPLIDA',
    ],
}


def cargar_datos_prueba(carpeta=None):
    """Escribe los seis archivos con los datos de prueba (reemplaza los actuales).

    Sin argumento usa RUTA_BASE. Retorna un diccionario con la cantidad de registros.
    """
    base = carpeta or RUTA_BASE
    os.makedirs(base, exist_ok=True)
    for nombre, lineas in DATOS_PRUEBA.items():
        ArchivoUtil.reescribir(os.path.join(base, nombre), lineas)
    return {n.replace('.txt', ''): len(l) for n, l in DATOS_PRUEBA.items()}

### Celda 5.2: pruebas de integración
`ejecutar_pruebas` recorre el flujo completo sobre una carpeta temporal, sin tocar los datos reales.

In [67]:
import tempfile


def _verificar(condicion, mensaje):
    """Lanza AssertionError con el mensaje si la condición es falsa."""
    if not condicion:
        raise AssertionError(mensaje)


def _debe_fallar(funcion, *args, contiene=''):
    """Verifica que la función lance ValueError (con un texto dado) y retorna el mensaje."""
    try:
        funcion(*args)
    except ValueError as e:
        _verificar(contiene in str(e), f'Mensaje inesperado: "{e}" (se esperaba "{contiene}")')
        return str(e)
    raise AssertionError('Se esperaba un ValueError y no se lanzó.')


def ejecutar_pruebas():
    """Ejecuta las 13 pruebas de integración. Retorna (todo_ok, informe_en_texto)."""
    informe = []
    fallos = [0]
    ctx = {}
    EST, DOC, EXT = TipoUsuarioEnum.ESTUDIANTE, TipoUsuarioEnum.DOCENTE, TipoUsuarioEnum.EXTERNO

    with tempfile.TemporaryDirectory(prefix='sgb_pruebas_') as carpeta:
        s = crear_servicios(carpeta)
        carpeta_muestra = os.path.join(carpeta, 'muestra')

        def correr(numero, descripcion, funcion):
            try:
                funcion()
                informe.append(f'OK      Prueba {numero:>2}: {descripcion}')
            except Exception as e:
                fallos[0] += 1
                informe.append(f'FALLÓ   Prueba {numero:>2}: {descripcion} -> {type(e).__name__}: {e}')

        def t1():
            s.libro.registrar_categoria('LIT', 'Literatura', 'Novelas')
            s.libro.registrar_categoria('TEC', 'Tecnología', 'Programación')
            s.libro.registrar('111', 'Libro Uno', 'Autor A', 'LIT', 2000, 2)
            s.libro.registrar('222', 'Libro Dos', 'Autor B', 'TEC', 2010, 1)
            s.libro.registrar('333', 'Libro Tres', 'Autor C', 'TEC', 2015, 3)
            for doc, nombre, tipo in [('1', 'Ana', EST), ('2', 'Beto', DOC), ('3', 'Caro', EXT),
                                      ('4', 'Dani', EST), ('5', 'Eli', EST)]:
                s.usuario.registrar(doc, nombre, tipo, f'{nombre}@correo.com', '300')
            _debe_fallar(s.libro.registrar_categoria, 'LIT', 'Otra', 'x', contiene='Ya existe')
            _debe_fallar(s.libro.registrar, '111', 'Dup', 'A', 'LIT', 2000, 1, contiene='Ya existe')
            _debe_fallar(s.usuario.registrar, '1', 'Dup', EST, 'a', 'b', contiene='Ya existe')
            _verificar(len(s.usuario.listar_todos()) == 5, 'Debe haber 5 usuarios.')
            _verificar(len(s.libro.listar_todos()) == 3, 'Debe haber 3 libros.')

        def t2():
            p = s.prestamo.prestar('1', '111', date(2025, 3, 1))
            _verificar(p.fecha_limite == '2025-03-08', 'Estudiante: límite a 7 días.')
            _verificar(s.libro.buscar_por_isbn('111').disponibles == 1, 'Debe quedar 1 ejemplar.')
            ctx['p_ana'] = p

        def t3():
            ctx['p_beto'] = s.prestamo.prestar('2', '222', date(2025, 3, 1))
            _verificar(s.libro.buscar_por_isbn('222').disponibles == 0, 'Libro 222 sin ejemplares.')
            r = s.reserva.reservar('1', '222', date(2025, 3, 2))
            _verificar(r.estado == EstadoReservaEnum.ACTIVA, 'La reserva debe quedar ACTIVA.')
            _debe_fallar(s.reserva.reservar, '1', '222', contiene='ya tiene una reserva')
            _debe_fallar(s.reserva.reservar, '3', '111', contiene='ejemplares disponibles')

        def t4():
            p, retraso, multa = s.prestamo.devolver(ctx['p_ana'].codigo, date(2025, 3, 11))
            _verificar(retraso == 3, f'Retraso esperado 3, fue {retraso}.')
            _verificar(multa is not None and multa.valor == 3000, 'La multa debe ser COP 3000.')
            _verificar(multa.estado == EstadoMultaEnum.PENDIENTE, 'La multa nace PENDIENTE.')
            _verificar(s.libro.buscar_por_isbn('111').disponibles == 2, 'Debe volver el ejemplar.')
            ctx['multa'] = multa

        def t5():
            _debe_fallar(s.prestamo.prestar, '1', '333', date(2025, 3, 12), contiene='multas pendientes')

        def t6():
            _p, retraso, multa = s.prestamo.devolver(ctx['p_beto'].codigo, date(2025, 3, 5))
            _verificar(retraso == 0 and multa is None, 'Devolución a tiempo sin multa.')
            _verificar(s.libro.buscar_por_isbn('222').disponibles == 1, 'Libro 222 con 1 ejemplar.')
            _debe_fallar(s.prestamo.prestar, '5', '222', date(2025, 3, 12), contiene='turno')

        def t7():
            pagada = s.multa.pagar(ctx['multa'].codigo)
            _verificar(pagada.estado == EstadoMultaEnum.PAGADA, 'La multa debe quedar PAGADA.')
            _debe_fallar(s.multa.pagar, ctx['multa'].codigo, contiene='ya fue pagada')
            s.prestamo.prestar('1', '222', date(2025, 3, 12))
            reservas = s.reserva.listar_por_usuario('1')
            _verificar(reservas[0].estado == EstadoReservaEnum.CUMPLIDA, 'Reserva CUMPLIDA.')
            _verificar(s.reserva.cola_libro('222') == [], 'La cola debe quedar vacía.')

        def t8():
            s.prestamo.prestar('3', '111', date(2025, 3, 1))
            s.prestamo.prestar('3', '333', date(2025, 3, 1))
            _debe_fallar(s.prestamo.prestar, '3', '333', date(2025, 3, 1), contiene='cupo de 2')

        def t9():
            s.usuario.actualizar('4', estado=ESTADO_USUARIO_SUSPENDIDO)
            _debe_fallar(s.prestamo.prestar, '4', '333', date(2025, 3, 1), contiene='no está activo')
            _debe_fallar(s.reserva.reservar, '4', '222', contiene='no está activo')

        def t10():
            corte = date(2025, 4, 1)
            _verificar(len(s.prestamo.listar_vencidos(corte)) == 3, 'Deben ser 3 vencidos.')
            marcados = s.prestamo.marcar_vencidos(corte)
            _verificar(len(marcados) == 3, 'Deben marcarse 3 préstamos.')
            estados = [p.estado for p in s.prestamo.listar_todos()]
            _verificar(estados.count(EstadoPrestamoEnum.VENCIDO) == 3, 'Deben quedar 3 VENCIDO guardados.')

        def t11():
            cargar_datos_prueba(carpeta_muestra)
            m = crear_servicios(carpeta_muestra)
            m.prestamo.marcar_vencidos(date(2025, 11, 21))
            top = m.analisis.libros_mas_prestados(n=5)
            _verificar([f[2] for f in top] == [5, 3, 1, 1, 1], f'Ranking inesperado: {top}')
            _verificar(top[0][1] == 'Cien años de soledad', 'El más prestado es Cien años de soledad.')
            cat = {f[1]: (f[2], f[3]) for f in m.analisis.prestamos_por_categoria()}
            _verificar(cat == {'Tecnología': (5, 41.7), 'Literatura': (5, 41.7),
                               'Ciencias': (1, 8.3), 'Historia': (1, 8.3)}, f'Categorías: {cat}')
            mor = {f[0]: f[1:] for f in m.analisis.morosidad_por_tipo()}
            _verificar(mor == {'ESTUDIANTE': (3, 6, 50.0), 'DOCENTE': (2, 4, 50.0),
                               'EXTERNO': (2, 2, 100.0)}, f'Morosidad: {mor}')
            prom, n = m.analisis.duracion_promedio_prestamo()
            _verificar(n == 8 and abs(prom - 8.75) < 1e-9, f'Duración: {prom} sobre {n}.')
            _verificar(m.analisis.reporte_multas() == (11000, 8000, 3000), 'Reporte de multas.')

        def t12():
            s2 = crear_servicios(carpeta)
            _verificar(len(s2.usuario.listar_todos()) == 5, 'Usuarios persistidos.')
            _verificar(len(s2.libro.listar_todos()) == 3, 'Libros persistidos.')
            _verificar(len(s2.prestamo.listar_todos()) == 5, 'Préstamos persistidos.')
            _verificar(len(s2.multa.listar_todas()) == 1, 'Multas persistidas.')
            _verificar(len(s2.reserva.listar_todas()) == 1, 'Reservas persistidas.')

        def t13():
            cargar_datos_prueba(carpeta_muestra)
            m = crear_servicios(carpeta_muestra)
            cantidades = (len(m.libro.listar_categorias()), len(m.usuario.listar_todos()),
                          len(m.libro.listar_todos()), len(m.prestamo.listar_todos()),
                          len(m.multa.listar_todas()), len(m.reserva.listar_todas()))
            _verificar(cantidades == (4, 6, 6, 12, 3, 2), f'Cantidades: {cantidades}')
            usuarios = {u.num_documento for u in m.usuario.listar_todos()}
            libros = {l.isbn: l for l in m.libro.listar_todos()}
            prestamos = m.prestamo.listar_todos()
            for p in prestamos:
                _verificar(p.num_documento in usuarios and p.isbn in libros, f'Referencia rota: {p.codigo}')
            for mu in m.multa.listar_todas():
                _verificar(mu.valor == mu.dias_retraso * MULTA_POR_DIA, f'Valor de {mu.codigo}.')
                _verificar(any(p.codigo == mu.cod_prestamo for p in prestamos), f'Préstamo de {mu.codigo}.')
            for r in m.reserva.listar_todas():
                _verificar(r.num_documento in usuarios and r.isbn in libros, f'Referencia de {r.codigo}.')
            for isbn, l in libros.items():
                abiertos = sum(1 for p in prestamos
                               if p.isbn == isbn and p.estado != EstadoPrestamoEnum.DEVUELTO)
                _verificar(l.disponibles == l.total_ejemplares - abiertos, f'Disponibles de {l.titulo}.')

        correr(1, 'Registro de categorías, libros y usuarios; duplicados rechazados', t1)
        correr(2, 'Préstamo de estudiante a 7 días y descuento de un ejemplar', t2)
        correr(3, 'Reserva de un libro sin ejemplares; reserva repetida rechazada', t3)
        correr(4, 'Devolución con 3 días de retraso genera multa de COP 3000', t4)
        correr(5, 'Usuario con multa pendiente no puede prestar', t5)
        correr(6, 'El libro reservado solo lo presta el primero en espera', t6)
        correr(7, 'Pago de la multa y préstamo al usuario con reserva (queda CUMPLIDA)', t7)
        correr(8, 'Cupo de 2 libros para un usuario externo', t8)
        correr(9, 'Usuario suspendido no puede prestar ni reservar', t9)
        correr(10, 'Detección de 3 préstamos vencidos y cambio a VENCIDO', t10)
        correr(11, 'Los cinco indicadores de análisis con los valores esperados', t11)
        correr(12, 'Persistencia: los datos siguen al recrear los servicios', t12)
        correr(13, 'Datos de prueba cargados y consistentes', t13)

    resumen = ('RESULTADO: las 13 pruebas pasaron.' if fallos[0] == 0
               else f'RESULTADO: {fallos[0]} prueba(s) fallaron.')
    informe.append(resumen)
    return fallos[0] == 0, '\n'.join(informe)

### Celda 5.3: ejecución principal
`CARGAR_DATOS_DE_PRUEBA = True` reemplaza los archivos de Drive con los datos de prueba cada vez que se ejecuta la celda. Cambiar a `False` para trabajar con los datos reales.

In [68]:
CARGAR_DATOS_DE_PRUEBA = True

if CARGAR_DATOS_DE_PRUEBA:
    resumen = cargar_datos_prueba()
    print('Datos de prueba cargados en', RUTA_BASE)
    print(resumen)

ok, informe = ejecutar_pruebas()
print(informe)

demo = construir_interfaz()
demo.launch(debug=False)

Datos de prueba cargados en /content/drive/MyDrive/SGB/datos/
{'categorias': 4, 'usuarios': 6, 'libros': 6, 'prestamos': 12, 'multas': 3, 'reservas': 2}
OK      Prueba  1: Registro de categorías, libros y usuarios; duplicados rechazados
OK      Prueba  2: Préstamo de estudiante a 7 días y descuento de un ejemplar
OK      Prueba  3: Reserva de un libro sin ejemplares; reserva repetida rechazada
OK      Prueba  4: Devolución con 3 días de retraso genera multa de COP 3000
OK      Prueba  5: Usuario con multa pendiente no puede prestar
OK      Prueba  6: El libro reservado solo lo presta el primero en espera
OK      Prueba  7: Pago de la multa y préstamo al usuario con reserva (queda CUMPLIDA)
OK      Prueba  8: Cupo de 2 libros para un usuario externo
OK      Prueba  9: Usuario suspendido no puede prestar ni reservar
OK      Prueba 10: Detección de 3 préstamos vencidos y cambio a VENCIDO
OK      Prueba 11: Los cinco indicadores de análisis con los valores esperados
OK      Prueba 12: Pers